# SCFL Gas–Electric Seam: Tuning Package 1 (gating rule v4-Tune-2), tuning set only, not blind
Upstream Coherence Measurement Stratum (UCMS) | Ronald L. Brogdon | ORCID: https://orcid.org/0009-0009-0507-2971

Frozen engine v1.1.0 (https://doi.org/10.5281/zenodo.23839571). Commit and scenarios in `inputs/PROVENANCE.json`. These runs belong to the tuning set and are never part of the sealed test.

Every input is 24 hours: 00:00–06:00 is the settled pre-window (SHIFT baseline), the test window runs 06:00–24:00, and strain starts at 07:00.

12 conditions × 30 seeds = 360 inputs (INPUT_<condition>_s01…s30):
- T01 control
- T02–T04 strain only, low / medium / high (21% / 41% / 54% of processing supply lost, ramped 07:00–09:00)
- T05 strain medium, slow ramp (07:00–13:00); T06 strain medium, fast ramp (07:00–07:15)
- T07 medium strain + Houston station c15 isolated 08:00
- T08 medium strain + Victoria bypassed 08:00 (mild break)
- T09 medium strain + Dallas-east station isolated 08:00 (far)
- T10 medium strain + Permian station isolated 08:00 (very far)
- T11 medium strain + Houston station c20 isolated 07:15 (before the gate)
- T12 medium strain + Victoria isolated 10:00 (well after the gate)

**Operator settings are not in the package.** To use your own, upload them in Cell 1 as `scfl_private_settings.json` (same structure as `scan_config.json`, only the keys you want to change). Cell 3 merges them into each input's config inside this Colab session only. Without that file, the engine runs on the package defaults.

1. **Upload** `scfl_measure.py`, `scfl_ingest_qc.py`, `SCFL_tune1_inputs.zip` and, optionally, `scfl_private_settings.json`.
2. **Check** the engine is the frozen v1.1.0.
3. **Run** ingest QC and the engine on all 360 inputs (about 30 minutes).
4. **Save** fingerprint the outputs and download them.

In [ ]:
# Cell 1: upload the two engine files, the inputs zip and (optional) scfl_private_settings.json
import os, glob, shutil
from google.colab import files
files.upload()
# Fix names the browser changed, e.g. 'scfl measure (1).py' or 'scfl measure.py'
want = {'scfl_measure.py': 'measure', 'scfl_ingest_qc.py': 'ingest', 'SCFL_tune1_inputs.zip': 'tune1'}
for name, key in want.items():
    if os.path.exists(name):
        continue
    found = [f for f in glob.glob('*') if key in f.lower().replace(' ', '_').replace('_(','(') and f != name
             and f.lower().endswith(name.rsplit('.', 1)[1])]
    if found:
        shutil.copy(sorted(found)[-1], name); print('renamed', sorted(found)[-1], '->', name)
missing = [n for n in want if not os.path.exists(n)]
print('Missing:', missing) if missing else print('All three files present. Run Cell 2.')
pv = [f for f in glob.glob('*') if 'private' in f.lower() and f.lower().endswith('.json')]
if pv and not os.path.exists('scfl_private_settings.json'):
    shutil.copy(sorted(pv)[-1], 'scfl_private_settings.json')
print('Private settings:', 'found' if os.path.exists('scfl_private_settings.json') else 'none (package defaults)')

In [ ]:
# Cell 2: fix any browser-renamed files, then confirm the frozen engine (line endings normalized, as in the freeze)
import hashlib, os, glob, shutil
for name, key in {'scfl_measure.py': 'measure', 'scfl_ingest_qc.py': 'ingest', 'SCFL_tune1_inputs.zip': 'tune1'}.items():
    if not os.path.exists(name):
        found = [f for f in glob.glob('*') if key in f.lower().replace(' ', '_').replace('_(','(') and f.lower().endswith(name.rsplit('.', 1)[1])]
        if found: shutil.copy(sorted(found)[-1], name); print('renamed', sorted(found)[-1], '->', name)
def h(p): return hashlib.sha256(open(p,'rb').read().replace(b'\r\n', b'\n')).hexdigest()
FROZEN = {'scfl_measure.py': '233d9d9c154639c3e7c4fcc0f57bc2044cfe313e17f89d09bed9610a61e5b020',
          'scfl_ingest_qc.py': '50a9ebc2a860fe0a8b82eb7d716e0dce13432bda89ea0fb5768d535ccb8eaa3b'}
ok = True
for f, want_h in FROZEN.items():
    got = h(f); print(f, 'MATCH' if got == want_h else 'MISMATCH ' + got); ok &= got == want_h
zh = hashlib.sha256(open('SCFL_tune1_inputs.zip','rb').read()).hexdigest()
print('inputs package', 'MATCH' if zh == 'fac2656dd4a9f54dd17eba9bbffab65d3bd73dcc51275176ae9b7bf0acc82434' else 'MISMATCH ' + zh)
print('Frozen engine confirmed. Run Cell 3.' if ok else 'STOP: engine does not match the freeze.')

In [ ]:
# Cell 3: run ingest QC and the engine on every input
import zipfile, subprocess, json
zipfile.ZipFile('SCFL_tune1_inputs.zip').extractall('.')
def merge(a, b):
    for k, v in b.items():
        if isinstance(v, dict) and isinstance(a.get(k), dict): merge(a[k], v)
        else: a[k] = v
    return a
PRIV = json.load(open('scfl_private_settings.json')) if os.path.exists('scfl_private_settings.json') else None
if PRIV:
    for d in glob.glob('inputs/INPUT_*'):
        cfg = json.load(open(f'{d}/scan_config.json')); json.dump(merge(cfg, PRIV), open(f'{d}/scan_config.json', 'w'), indent=2)
    print('private settings merged into', len(glob.glob('inputs/INPUT_*')), 'configs (this session only)')
ING, ENG = os.path.abspath('scfl_ingest_qc.py'), os.path.abspath('scfl_measure.py')
for d in sorted(glob.glob('inputs/INPUT_*')):
    q = subprocess.run(['python', ING, 'series_manifest.csv', '--out', 'qc_output'], cwd=d, capture_output=True, text=True)
    try:
        verdict = json.load(open(f'{d}/qc_output/qc_summary.json'))['run']['verdict']
    except Exception:
        print(d, 'INGEST ERROR:', (q.stderr or q.stdout)[-600:]); continue
    print(d, 'ingest:', verdict)
    if verdict != 'READY':
        print((q.stdout or '')[-600:]); continue
    r = subprocess.run(['python', ENG, 'qc_output/aligned_series.csv', 'series_manifest.csv',
                        '--config', 'scan_config.json', '--out', 'measure_output'], cwd=d, capture_output=True, text=True)
    print(d, 'engine:', 'RUN' if r.returncode == 0 else 'ENGINE ERROR: ' + (r.stderr or r.stdout)[-600:])
print('done:', len(glob.glob('inputs/INPUT_*/measure_output/operators_summary.json')), 'of', len(glob.glob('inputs/INPUT_*')), 'inputs measured')

In [ ]:
# Cell 4: fingerprint the outputs and download them
import os, glob, zipfile, hashlib
from google.colab import files
out = 'tune1_readings.zip'
lines = []
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob('inputs/INPUT_*/qc_output/*') + glob.glob('inputs/INPUT_*/measure_output/*')):
        if os.path.isfile(f):
            z.write(f); lines.append(f'{hashlib.sha256(open(f,"rb").read()).hexdigest()}  {f}')
open('tune1_fingerprints.txt', 'w').write('\n'.join(lines) + '\n')
print(len(lines), 'output files')
files.download(out); files.download('tune1_fingerprints.txt')